In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATA_PATH = Path("../data/application_train.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (307511, 122)


In [2]:
# TARGET is what we want the model to predict.
# X = input features
# y = loan default outcome

X = df.drop(columns=["TARGET"]).copy()
y = df["TARGET"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (307511, 121)
y shape: (307511,)


In [3]:
# Applicant ID is an identifier, not a meaningful predictive feature.
# Keeping it could allow the model to learn meaningless patterns.

X = X.drop(columns=["SK_ID_CURR"])

print("X shape after removing ID:", X.shape)

X shape after removing ID: (307511, 120)


In [4]:
# Home Credit uses 365243 as a special value in DAYS_EMPLOYED.
# Treat it as missing rather than allowing the model to interpret
# it as an actual employment duration.

X["DAYS_EMPLOYED"] = X["DAYS_EMPLOYED"].replace(365243, np.nan)

print(
    "Missing DAYS_EMPLOYED after correction:",
    X["DAYS_EMPLOYED"].isna().sum()
)

Missing DAYS_EMPLOYED after correction: 55374


In [5]:
#validation split

from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set :", X_train.shape)
print("Validation set:", X_valid.shape)

Training set : (246008, 120)
Validation set: (61503, 120)


In [6]:
print("Training default rate :", y_train.mean())
print("Validation default rate:", y_valid.mean())

Training default rate : 0.08072908198107379
Validation default rate: 0.08072776937710356


In [7]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features  :", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numerical features  : 104
Categorical features: 16


In [8]:
#Build preprocessing pipelines

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=True
    ))
])

In [9]:
#combine pipelines

from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [10]:
#Fit the training data 

# IMPORTANT:
# Fit preprocessing only using the training data.
# This prevents information from the validation set leaking into training.

preprocessor.fit(X_train)

print("Preprocessor fitted successfully.")

Preprocessor fitted successfully.


In [11]:
#Transfor train vaildation data

X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print("Processed training shape  :", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)

Processed training shape  : (246008, 244)
Processed validation shape: (61503, 244)


In [12]:
print("Missing values in processed training data:")

print(
    np.isnan(X_train_processed.toarray()).sum()
    if hasattr(X_train_processed, "toarray")
    else np.isnan(X_train_processed).sum()
)

Missing values in processed training data:
0


In [13]:
if hasattr(X_train_processed, "data"):
    print(
        "NaN values:",
        np.isnan(X_train_processed.data).sum()
    )
else:
    print(
        "NaN values:",
        np.isnan(X_train_processed).sum()
    )

NaN values: 0


In [14]:
#Save the preprocessing object

import joblib

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    preprocessor,
    MODEL_DIR / "baseline_preprocessor.joblib"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.
